# EOG / EMG drowsiness detection (eye-electrode branch)

Detection from eye electrodes, in two parts that share the same code:
* **Part A, pre-recorded data**: replay the recordings of the DD-Database like a sensor, train the model, evaluate it on people it has never seen, and show the scores.
* **Part B, live data**: run the same detector on a live sensor (your EMG sensor), or on a simulated one until it arrives.

(The exploration of the data itself is in `EOG_EDA.ipynb`.)

```
sensor samples ─► filter ─► eye-event detector ─► 4 features over the last 120 s ─► model ─► {score, confidence, raw, timestamp}
(file or live)              (blinks / eye movements)   (rate, width, closure share, regularity)   (logistic regression)
```
The branch has the same output contract as the camera branches, so it can join the fusion later.

**What to expect (read this first).**
* No pretrained model exists for this job (section 3 says why), so a small, explainable model is **trained from scratch**.
* The labels are the driver's own button presses ("I feel drowsy now"). They are noisy and there are only 7 people, so the result is a **useful but weak** signal (AUC about 0.6-0.7, where 0.5 is a coin flip). Part A states it again with the numbers.
* The EMG sensor is a **different sensor** from the EOG used for training. The code for it is complete and tested on simulated signals, but it has **not been validated on real EMG data**. Part B says what to check when the sensor arrives.
* Part B needs the trained model file from Part A (`models/eog_drowsiness_model.json`); run Part A once first.

## 1. Setup and data

In [1]:
import io, json, os, time, urllib.request
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyedflib
from IPython.display import Image, display
from scipy import signal
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = "data/dd_database"
MODEL_PATH = "models/eog_drowsiness_model.json"
ZENODO = "https://zenodo.org/api/records/8284057/files/{}/content"
RECORDINGS = ["01M_1", "02F_1", "03F_1", "04M_1", "05M_1", "06M_1", "07F_1"]    # trial 1 of subjects 1-7 (91 MB in total)
FS = 128                                  # sampling rate of the recordings (Hz)

CALIBRATE_S = 30       # the detector learns the person's noise level
BASELINE_S = 300       # then 5 minutes of the person's normal (alert) eye activity are measured ...
WINDOW_S = 120         # ... and every 120 s window is compared with that baseline
STEP_S = 30            # a new score every 30 s
FEATURES = ["rate", "width", "share", "ivcv"]      # events per minute, mean event length, closure share, irregularity

**Dataset: DD-Database** (Drivers Drowsiness Database, Universidad Nacional de San Juan, [Zenodo](https://zenodo.org/records/8284057), CC0). 10 volunteers drove a **night-time driving simulator** for 2 hours (twice), partly sleep-deprived, and pressed a button whenever they **felt drowsy**. Recorded: 2 EOG channels (LOC, ROC), 4 EEG, 1 ECG, 128 Hz, raw. Only trial 1 of subjects 1-7 is used (7 files, 91 MB); subjects 8-10 pressed the button only 2-8 times in total, so they carry almost no drowsy examples.

In [2]:
os.makedirs(DATA_DIR, exist_ok=True)
for name in RECORDINGS:
    for suffix in ("", "_annotations"):
        path = f"{DATA_DIR}/{name}{suffix}.edf"
        if not os.path.exists(path):
            print("downloading", path)
            urllib.request.urlretrieve(ZENODO.format(f"{name}{suffix}.edf"), path)


def read_recording(name):
    """Left and right eye channel (microvolts) and the times (s) of the driver's 'I feel drowsy' button presses."""
    r = pyedflib.EdfReader(f"{DATA_DIR}/{name}.edf")
    labels = r.getSignalLabels()
    left, right = r.readSignal(labels.index("LOC-Ref")), r.readSignal(labels.index("ROC-Ref"))
    r.close()
    a = pyedflib.EdfReader(f"{DATA_DIR}/{name}_annotations.edf")
    press = np.array(a.readAnnotations()[0])
    a.close()
    return left, right, press[(press > 30) & (press < 7000)]   # the marks at about 10 s and 7220 s are start / end markers


data = {name: read_recording(name) for name in RECORDINGS}
pd.DataFrame({name: {"length (min)": len(l) / FS / 60, "button presses": len(p), "first press (min)": p[0] / 60}
              for name, (l, r_, p) in data.items()}).T.round(1)

,length (min),button presses,first press (min)
01M_1,121.8,23.0,25.2
02F_1,123.0,34.0,3.8
03F_1,121.8,81.0,13.2
04M_1,115.5,35.0,6.6
05M_1,120.8,23.0,18.4
06M_1,120.2,33.0,12.9
07F_1,121.5,88.0,13.0


## 2. Finding eye events

An **eye event** is a short, fast deflection of the eye signal: a blink or a quick eye movement. Slow drifts are not events. The detector works on a stream (sample by sample), so exactly the same code runs on a recording and on a live sensor:

1. Filter the signal (EOG: band-pass 0.1-20 Hz and take the absolute value; EMG: see Part B).
2. Measure everything in units of the signal's **own noise** (robust median and MAD over the last minute). No fixed microvolt limits, so gain, units and the person's signal size do not matter.
3. An event **starts** when the signal is above 5 noise units **and rising steeply** (this rejects slow movements) and **ends** when it falls below 30% of its own peak. Two events closer than 0.2 s are one.
4. Every sample also gets a **confidence**: 0 while the noise level is still being learned, and low if the signal is flat (electrode off), clipped (amplifier saturated), unusually noisy (motion), or the event rate is not believable.

In [3]:
MIN_CONF = 0.3            # below this a sample counts as UNKNOWN, not OPEN / CLOSED


def clip01(x):
    return float(np.clip(x, 0.0, 1.0))


class StreamFilter:
    """A causal filter that keeps its state between chunks."""

    def __init__(self, sos):
        self.sos, self.zi = sos, None

    def __call__(self, x):
        if len(x) == 0:
            return x
        if self.zi is None:
            self.zi = signal.sosfilt_zi(self.sos) * x[0]
        y, self.zi = signal.sosfilt(self.sos, x, zi=self.zi)
        return y


class BlinkDetector:
    """Activity signal in -> per-sample state (OPEN / CLOSED / UNKNOWN) and confidence out; finished events in self.events."""

    def __init__(self, fs, calibrate_s, z_on=5.0, slope_on=4.0, slope_win_s=0.10, z_off=2.0, end_frac=0.3,
                 max_event_s=20.0, sigma_window_s=60.0, merge_gap_s=0.2):
        self.fs, self.calibrate_n = fs, int(calibrate_s * fs)
        self.z_on, self.slope_on, self.z_off, self.end_frac = z_on, slope_on, z_off, end_frac
        self.max_event_s, self.merge_gap_s = max_event_s, merge_gap_s
        self.ring = deque(maxlen=int(sigma_window_s * fs))                  # last minute: median / MAD
        self.recent = deque(maxlen=max(1, int(round(slope_win_s * fs))) + 1)
        self.diffs = deque(maxlen=self.ring.maxlen)                         # sample-to-sample changes: fast noise
        self.short_diffs = deque(maxlen=int(2 * fs))
        self._prev, self.count, self.since_update = None, 0, 0
        self.median, self.sigma, self.sigma_cal = 0.0, 0.0, None
        self.diff_sigma, self.diff_short, self.diff_cal = 0.0, 0.0, None
        self.in_event, self.peak, self.start_t = False, 0.0, 0.0
        self.events, self._event_ends, self.last_factors = [], deque(maxlen=200), {}

    @property
    def calibrated(self):
        return self.sigma_cal is not None

    def _update_stats(self):
        data = np.asarray(self.ring)
        self.median = float(np.median(data))
        self.sigma = 1.4826 * float(np.median(np.abs(data - self.median)))
        mad = lambda v: 1.4826 * float(np.median(np.abs(v - np.median(v))))
        self.diff_sigma = mad(np.asarray(self.diffs)) if len(self.diffs) > 10 else 0.0
        self.diff_short = mad(np.asarray(self.short_diffs)) if len(self.short_diffs) > 10 else self.diff_sigma
        if self.sigma_cal is None and self.count >= self.calibrate_n and self.sigma > 1e-9 * (abs(self.median) + 1.0):
            self.sigma_cal, self.diff_cal = self.sigma, self.diff_sigma     # (a flat line does not count as calibrated)

    def _factors(self, t, flags):
        f = {"calibrated": 1.0 if self.calibrated else 0.0, "clipped": 0.0 if flags["clipped"] else 1.0}
        if self.calibrated:
            tiny = 1e-9 * (abs(self.median) + 1.0)
            ratio = self.diff_short / max(self.diff_cal, tiny)              # fast changes now vs at calibration
            f["flat"] = 0.0 if (flags["flat"] or self.sigma <= tiny) else clip01((ratio - 0.1) / 0.2)
            f["noise"] = 1.0 - clip01((ratio - 2.5) / 3.0)
            rate = sum(e > t - 30.0 for e in self._event_ends) * 2.0         # events per minute over the last 30 s
            f["rate"] = 1.0 - clip01((rate - 90.0) / 60.0)
        else:
            f["flat"], f["noise"], f["rate"] = (0.0 if flags["flat"] else 1.0), 1.0, 1.0
        return f

    def process(self, times, activity, flags):
        """One chunk of samples -> (states, confidences), one entry per sample."""
        states, confs = [], []
        factors = self._factors(times[0] if len(times) else 0.0, flags)
        conf = min(factors.values())
        for t, a in zip(times, activity):
            self.ring.append(a)
            if self._prev is not None:
                self.diffs.append(a - self._prev); self.short_diffs.append(a - self._prev)
            self._prev = a
            self.count += 1; self.since_update += 1
            if self.since_update >= self.fs or self.count == self.calibrate_n:
                self._update_stats(); self.since_update = 0
                factors = self._factors(t, flags); conf = min(factors.values())
            old = self.recent[0] if len(self.recent) == self.recent.maxlen else a
            self.recent.append(a)

            if not self.calibrated or self.sigma <= 0:
                states.append("UNKNOWN"); confs.append(0.0 if not self.calibrated else conf)
                continue
            z, dz = (a - self.median) / self.sigma, (a - old) / self.sigma
            if not self.in_event:
                if z >= self.z_on and dz >= self.slope_on:
                    self.in_event, self.peak, self.start_t = True, z, t
                    if self.events and t - self.events[-1]["end"] < self.merge_gap_s:       # a burst that dipped and rose again
                        last = self.events.pop()
                        self.peak, self.start_t = max(self.peak, last["peak_z"]), last["start"]
                        if self._event_ends and self._event_ends[-1] == last["end"]:
                            self._event_ends.pop()
            else:
                self.peak = max(self.peak, z)
                if z < max(self.z_off, self.end_frac * self.peak) or t - self.start_t > self.max_event_s:
                    self.in_event = False
                    self.events.append({"start": self.start_t, "end": t, "duration": t - self.start_t, "peak_z": self.peak})
                    if min(v for k, v in factors.items() if k != "rate") >= MIN_CONF:
                        self._event_ends.append(t)                  # events seen while the signal was bad do not count
            states.append("UNKNOWN" if conf < MIN_CONF else ("CLOSED" if self.in_event else "OPEN"))
            confs.append(conf)
        self.last_factors = factors
        return states, confs

### From sensor samples to events

`EyeEventStream` is the whole front half of the branch: raw microvolts in, events and confidence out. Only the filter differs between the two sensor types; everything after is shared.

In [4]:
class EyeEventStream:
    """Microvolts -> activity signal -> BlinkDetector. kind: "eog" (eye potentials) or "emg" (eyelid muscle activity)."""

    def __init__(self, fs, kind, mains_hz=50.0, calibrate_s=CALIBRATE_S):
        self.fs, self.kind, self.notch, self._skip = fs, kind, None, 0
        if kind == "eog":
            # blinks and eye movements are slow (0.1-20 Hz); the absolute value makes up and down movements alike
            self.bandpass = StreamFilter(signal.butter(2, [0.1, min(20.0, 0.4 * fs)], btype="band", fs=fs, output="sos"))
        elif kind == "emg":
            # muscle activity is fast noise (20-250 Hz): remove the mains hum, rectify, smooth to an envelope (~50 ms)
            if fs < 200:
                raise ValueError(f"EMG needs a sampling rate of at least 200 Hz (the device gives {fs} Hz)")
            self.bandpass = StreamFilter(signal.butter(4, [20.0, min(250.0, 0.45 * fs)], btype="band", fs=fs, output="sos"))
            if mains_hz < 0.45 * fs:
                self.notch = StreamFilter(signal.tf2sos(*signal.iirnotch(mains_hz, 30.0, fs)))
            self.smooth = StreamFilter(signal.butter(2, 20.0, btype="low", fs=fs, output="sos"))
        else:
            raise ValueError("kind must be 'eog' or 'emg'")
        self.k = max(1, int(fs // 200))                                    # work at about 200 Hz
        self.detector = BlinkDetector(fs / self.k, calibrate_s)
        self.trace = deque(maxlen=int(30 * fs / self.k))                   # last 30 s of activity, for plots

    def process(self, times, uV):
        times, uV = np.asarray(times, float), np.asarray(uV, float)
        flat = len(uV) > 1 and float(np.ptp(uV)) == 0.0                    # a perfectly flat chunk: electrode off
        clipped = len(uV) >= 8 and not flat and np.mean((uV == uV.max()) | (uV == uV.min())) > 0.4   # amplifier saturated
        y = self.bandpass(uV)
        if self.kind == "eog":
            activity = np.abs(y)
        else:
            y = self.notch(y) if self.notch else y
            activity = np.maximum(self.smooth(np.abs(y)), 0.0)
        idx = np.arange(self._skip, len(activity), self.k)                 # keep every k-th sample
        self._skip = (self._skip - len(activity)) % self.k
        self.trace.extend(zip(times[idx], activity[idx]))
        return self.detector.process(times[idx], activity[idx], {"flat": bool(flat), "clipped": bool(clipped)})

**Visual check** on subject 1: red = detected events. Left: a busy minute, right: a quiet one. The marks sit on the real fast deflections, and the slow wobbles between them are left alone.

In [5]:
left, right, press = data["01M_1"]
t = np.arange(len(left)) / FS
stream = EyeEventStream(FS, "eog")
stream.process(t, left - right)
events = stream.detector.events
print(f"{len(events)} events in {t[-1] / 60:.0f} minutes ({len(events) / (t[-1] / 60):.1f} per minute); noise level learned: {stream.detector.sigma_cal:.1f} uV")

fig, axes = plt.subplots(2, 1, figsize=(13, 5.5))
eye = signal.sosfilt(signal.butter(2, [0.1, 20], btype="band", fs=FS, output="sos"), left - right)
for ax, t0 in zip(axes, (1500, 3000)):
    m = (t >= t0) & (t < t0 + 30)
    ax.plot(t[m], eye[m], lw=0.7)
    for e in events:
        if t0 <= e["start"] < t0 + 30:
            ax.axvspan(e["start"], e["end"], color="r", alpha=0.35)
    ax.set_ylabel("uV"); ax.set_title(f"subject 1, {t0 // 60} min {t0 % 60} s: detected events in red")
axes[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

3201 events in 122 minutes (26.3 per minute); noise level learned: 12.5 uV


## 3. Is there a pretrained model? No, so one is trained from scratch

Checked first, as preferred:
* **`joncarter/wav2sleep-eog`** (Hugging Face, MIT) is the closest: it takes the same two EOG channels at about the same rate. But it classifies **sleep stages** from whole-night recordings with a non-causal network, not driver drowsiness, it cannot run live, it cannot take an **EMG** signal, and its code is not on PyPI (it would mean running an unvetted GitHub repository). Not suitable.
* No pretrained model for EMG or EOG drowsiness exists on Hugging Face (searches for EOG, EMG, blink, drowsiness).

So a **logistic regression** is trained on four explainable features per 120 s window. It is small enough to read, and it can be checked.

## 4. The monitor: features, confidence and the output contract

`DrowsinessMonitor` is what runs, on a recording or live: it takes sensor samples and returns `{score, confidence, raw, timestamp}`.

* **Features** over the last 120 s, **relative to the person's own baseline** (their first 5 minutes after calibration, assumed alert). Relative values matter: a different sensor or electrode placement changes absolute numbers, but "this person now produces half the events they did at the start" survives.
  `rate` events per minute, `width` mean event length, `share` fraction of time inside events (PERCLOS-like closure share), `ivcv` irregularity of the gaps between events.
* **Score** = model output in 0-1 (higher = more drowsy). **Confidence** = signal quality over the window (flat, clipped, noisy, no data all lower it), 0 until calibration and baseline are done.

In [6]:
class LinearModel:
    """A trained logistic regression, small enough to save as JSON (so live code and fusion need no scikit-learn)."""

    def __init__(self, mean, scale, coef, intercept, info=None):
        self.mean, self.scale, self.coef = np.asarray(mean), np.asarray(scale), np.asarray(coef)
        self.intercept, self.info = float(intercept), info or {}

    def score(self, rel):
        z = (np.array([rel[f] for f in FEATURES]) - self.mean) / self.scale
        return float(1 / (1 + np.exp(-(z @ self.coef + self.intercept))))

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        json.dump({"features": FEATURES, "mean": self.mean.tolist(), "scale": self.scale.tolist(), "coef": self.coef.tolist(),
                   "intercept": self.intercept, "info": self.info}, open(path, "w"), indent=1)

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        return cls(d["mean"], d["scale"], d["coef"], d["intercept"], d["info"])


def window_features(events, t_end, length):
    """The four features of the events that started in [t_end - length, t_end)."""
    e = [x for x in events if t_end - length <= x["start"] < t_end]
    if not e:
        return {"rate": 0.0, "width": 0.0, "share": 0.0, "ivcv": 0.0}
    d, s = np.array([x["duration"] for x in e]), np.array([x["start"] for x in e])
    gaps = np.diff(s)
    return {"rate": len(e) * 60 / length, "width": d.mean(), "share": d.sum() / length,
            "ivcv": gaps.std() / gaps.mean() if len(gaps) > 2 and gaps.mean() > 0 else 0.0}


class DrowsinessMonitor:
    """Sensor samples in -> {score, confidence, raw, timestamp} out. Works on a recording replayed in chunks and on a live sensor."""

    def __init__(self, model, fs, kind, to_microvolts=1.0, offset=0.0, polarity=1, mains_hz=50.0,
                 calibrate_s=CALIBRATE_S, baseline_s=BASELINE_S, window_s=WINDOW_S, max_gap_s=1.0):
        self.model, self.kind = model, kind
        self.to_microvolts, self.offset, self.polarity = to_microvolts, offset, polarity
        self.calibrate_s, self.baseline_s, self.window_s, self.max_gap_s = calibrate_s, baseline_s, window_s, max_gap_s
        self.stream = EyeEventStream(fs, kind, mains_hz, calibrate_s)
        self._chunks = deque()                         # (time of last sample, mean confidence of the chunk, samples)
        self._last_t, self._baseline = None, None

    def update(self, times, raw):
        """Feed a chunk of raw device values (and their times in seconds); returns the latest output."""
        uV = self.polarity * (np.asarray(raw, float) - self.offset) * self.to_microvolts      # device units -> microvolts
        states, confs = self.stream.process(times, uV)
        if len(confs):
            self._chunks.append((float(times[-1]), float(np.mean(confs)), len(confs)))
            self._last_t = float(times[-1])
            while self._chunks and self._chunks[0][0] < self._last_t - self.window_s - 5:
                self._chunks.popleft()
        return self.latest(self._last_t)

    def latest(self, now):
        det = self.stream.detector
        ready_at = self.calibrate_s + self.baseline_s + self.window_s
        raw = {"status": "ok", "kind": self.kind, "n_events": len(det.events), "long_closure_reliable": False,
               "ready_after_s": ready_at, "model_validated_for_this_sensor": self.kind == "eog"}
        if self._last_t is None or now - self._last_t > self.max_gap_s:
            return {"score": 0.0, "confidence": 0.0, "raw": {**raw, "status": "no data from the sensor"}, "timestamp": now}
        if self._last_t < ready_at:
            return {"score": 0.0, "confidence": 0.0, "timestamp": now,
                    "raw": {**raw, "status": f"learning your normal eye activity ({ready_at - self._last_t:.0f} s left)"}}
        if self._baseline is None:                                           # fixed once, from the alert start of the session
            self._baseline = window_features(det.events, self.calibrate_s + self.baseline_s, self.baseline_s)
        f = window_features(det.events, self._last_t, self.window_s)
        rel = {k: (f[k] / self._baseline[k] if self._baseline[k] > 0 else 1.0) for k in f}
        window = [c for c in self._chunks if c[0] > self._last_t - self.window_s]
        recent = [c for c in self._chunks if c[0] > self._last_t - 2.0]
        quality = float(np.average([c[1] for c in window], weights=[c[2] for c in window]))
        confidence = min(quality, float(np.mean([c[1] for c in recent])) if recent else 0.0)   # falls at once when the signal goes bad
        raw.update({"events_per_min": f["rate"], "mean_event_s": f["width"], "closure_share": f["share"], "irregularity": f["ivcv"],
                    "features_vs_baseline": rel, "signal_quality": quality, "confidence_factors": dict(det.last_factors)})
        score = self.model.score(rel) if self.model is not None else 0.0
        return {"score": score, "confidence": confidence, "raw": raw, "timestamp": now}

# Part A. Pre-recorded data

Replay the 7 recordings through the monitor as if they were live sensors, look at how the features behave when the driver feels drowsy, train the model and test it on unseen people.

## A1. Replay the recordings as live streams
Each recording is fed one second at a time, like a sensor. Every 30 s the monitor's features are stored (no model yet). The labels:
* **drowsy** = the driver pressed the button inside the window or in the 60 s after it;
* **alert** = no press from 3 minutes before to 3 minutes after the window;
* windows in between are ambiguous and are not used for training or scoring accuracy.

In [7]:
def label(press, t_end):
    after = press[press >= t_end - WINDOW_S] - t_end                       # presses inside the window or later
    if len(after) and after.min() <= 60:
        return 1
    return 0 if np.abs(press - (t_end - WINDOW_S / 2)).min() > 180 else -1


rows, events_of = [], {}
for name, (left, right, press) in data.items():
    monitor = DrowsinessMonitor(None, FS, "eog")
    diff = left - right                                                    # the eye signal
    for i in range(0, len(diff), FS):                                      # one second at a time
        t_chunk = np.arange(i, min(i + FS, len(diff))) / FS
        out = monitor.update(t_chunk, diff[i:i + FS])
        t_end = t_chunk[-1] + 1 / FS
        if abs(t_end % STEP_S) < 1e-6 and out["raw"]["status"] == "ok":
            rows.append({"rec": name, "subj": name[:2], "t_end": t_end, "confidence": out["confidence"], "label": label(press, t_end),
                         **out["raw"]["features_vs_baseline"]})
    events_of[name] = monitor.stream.detector.events
table = pd.DataFrame(rows)
print(f"{len(table)} windows from {table.subj.nunique()} people; mean confidence {table.confidence.mean():.2f}")
labelled = table[(table.label >= 0) & (table.confidence >= 0.5)].reset_index(drop=True)
print(f"labelled windows: {len(labelled)}  (drowsy {int(labelled.label.sum())}, alert {int((1 - labelled.label).sum())})")

1582 windows from 7 people; mean confidence 0.92
labelled windows: 1316  (drowsy 857, alert 459)


## A2. What does drowsy look like in the features?

**Eye events become rarer when the driver feels drowsy.** Below: the event rate relative to the person's own start, grouped by how close the next button press is. If the pattern is real, the rate should be lower near presses.

In [8]:
t_next = []
for _, row in table.iterrows():
    press = data[row.rec][2]
    ahead = press[press >= row.t_end - WINDOW_S] - row.t_end
    t_next.append(ahead.min() if len(ahead) else np.inf)
table["to_next_press"] = t_next
bins = [(-1, 60, "press within 60 s"), (60, 180, "60-180 s"), (180, 600, "3-10 min"), (600, 1e9, "more than 10 min")]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (feature, title) in zip(axes, [("rate", "event rate (events/min vs own start)"), ("width", "mean event length vs own start"), ("share", "closure share vs own start")]):
    means = [table[(table.to_next_press > a) & (table.to_next_press <= b)][feature].mean() for a, b, _ in bins]
    sem = [table[(table.to_next_press > a) & (table.to_next_press <= b)][feature].sem() for a, b, _ in bins]
    ax.bar([lab for _, _, lab in bins], means, yerr=sem, color=["#c44", "#e99", "#9bd", "#69c"]); ax.axhline(1, color="k", lw=0.8, ls="--")
    ax.set_title(title, fontsize=10); ax.tick_params(axis="x", labelsize=8, rotation=20)
plt.tight_layout(); plt.show()

**How informative is each feature on its own?** AUC for telling "drowsy" from "alert" windows, per person (0.5 = no information, 1 = perfect). A value **below** 0.5 means the feature goes **down** when drowsy.

In [9]:
single = {f: [roc_auc_score(g.label, g[f]) for _, g in labelled.groupby("subj") if g.label.nunique() == 2] for f in FEATURES}
single_table = pd.DataFrame({f: {"mean AUC": np.mean(v), "worst person": np.min(v), "best person": np.max(v)} for f, v in single.items()}).T
display(single_table.round(2))
fig, ax = plt.subplots(figsize=(8, 3))
ax.boxplot(list(single.values()), tick_labels=list(single)); ax.axhline(0.5, color="r", ls="--", lw=0.8); ax.set_ylabel("AUC per person"); ax.set_title("single-feature AUC across the 7 people (red: no information)")
plt.show()

,mean AUC,worst person,best person
rate,0.34,0.25,0.41
width,0.52,0.38,0.65
share,0.39,0.31,0.51
ivcv,0.56,0.48,0.63


**Insights from the exploration**
1. **Eye activity goes down when the driver feels drowsy.** Relative to the person's own start, the event rate and the closure share are lowest in the windows right before a button press and highest when no press is near (see the bars). The single-feature AUCs agree: **rate and closure share are well below 0.5**, meaning they are *lower* in drowsy windows. That matches the literature: drowsy people blink and look around less.
2. **Event length alone says almost nothing** (AUC close to 0.5), and **irregularity** is only slightly informative. No single feature is strong, which is why a combination is used.
3. **People differ a lot**: the worst and best person differ widely for every feature. Using each person's own baseline removes part of this, not all.
4. The labels are the drivers' own button presses, so even a perfect detector could not score perfectly: part of the "error" is noise in the labels.

## A3. Training and honest evaluation (leave one person out)

A model is trained on 6 people and tested on the 7th, seven times, so **every score below is for a person the model has never seen**. Mean and spread over people are reported, plus recall for each class and the confusion matrix, as accuracy alone would hide a model that favours one class.

In [10]:
def fit(df):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", C=0.3)).fit(df[FEATURES], df.label)
    scaler, clf = pipe[0], pipe[1]
    return LinearModel(scaler.mean_, scaler.scale_, clf.coef_[0], clf.intercept_[0])


per_person, oof = {}, {}
for person in labelled.subj.unique():
    model = fit(labelled[labelled.subj != person])
    test = labelled[labelled.subj == person].copy()
    test["score"] = [model.score(r) for _, r in test.iterrows()]
    oof[person] = test
    fpr, tpr, _ = roc_curve(test.label, test.score)
    per_person[person] = {"AUC": roc_auc_score(test.label, test.score), "fpr": fpr, "tpr": tpr,
                          "drowsy windows": int(test.label.sum()), "alert windows": int((1 - test.label).sum())}
result = pd.concat(oof.values())
aucs = np.array([v["AUC"] for v in per_person.values()])
print(f"AUC on unseen people: mean {aucs.mean():.2f}, spread (std) {aucs.std():.2f}, worst {aucs.min():.2f}, best {aucs.max():.2f}")
pd.DataFrame({p: {k: v for k, v in d.items() if k not in ("fpr", "tpr")} for p, d in per_person.items()}).T.round(2)

AUC on unseen people: mean 0.65, spread (std) 0.06, worst 0.57, best 0.74


,AUC,drowsy windows,alert windows
01,0.58,99.0,95.0
02,0.63,106.0,77.0
03,0.57,178.0,28.0
04,0.63,134.0,42.0
05,0.67,100.0,86.0
06,0.69,89.0,78.0
07,0.74,151.0,53.0


In [11]:
pred = (result.score > 0.5).astype(int)
cm = confusion_matrix(result.label, pred)
recall_alert, recall_drowsy = cm[0, 0] / cm[0].sum(), cm[1, 1] / cm[1].sum()
print(f"all unseen windows pooled: accuracy {np.mean(pred == result.label):.2f}, recall of ALERT {recall_alert:.2f}, recall of DROWSY {recall_drowsy:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for person, d in per_person.items():
    axes[0].plot(d["fpr"], d["tpr"], lw=1, label=f"{person} (AUC {d['AUC']:.2f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8); axes[0].legend(fontsize=7); axes[0].set_xlabel("false alarm rate"); axes[0].set_ylabel("detection rate"); axes[0].set_title("ROC per unseen person")
axes[1].bar(list(per_person), [d["AUC"] for d in per_person.values()], color="tab:blue"); axes[1].axhline(0.5, color="r", ls="--", lw=0.8); axes[1].axhline(aucs.mean(), color="k", lw=1)
axes[1].set_ylim(0.3, 1); axes[1].set_title(f"AUC per person (line: mean {aucs.mean():.2f})"); axes[1].set_xlabel("person left out")
im = axes[2].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[2].text(j, i, str(v), ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["alert", "drowsy"]); axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["alert", "drowsy"])
axes[2].set_xlabel("predicted"); axes[2].set_ylabel("driver said"); axes[2].set_title("confusion matrix (score > 0.5)")
plt.tight_layout(); plt.show()

all unseen windows pooled: accuracy 0.63, recall of ALERT 0.67, recall of DROWSY 0.60


**How to read these graphs.** *Left, ROC:* each curve is one unseen person; the further above the dashed diagonal, the better the score separates drowsy from alert windows. *Middle:* the area under that curve (AUC) per person; 0.5 is a coin flip. *Right, confusion matrix:* rows are what the driver said, columns what the model said with the cut-off 0.5; the diagonal is correct.

### The score over time for every unseen person
Black line = drowsiness score, red ticks at the top = the driver's button presses, orange band = windows labelled drowsy. A useful score rises before and around the presses.

In [12]:
all_scores = {}
for person in labelled.subj.unique():
    model = fit(labelled[labelled.subj != person])
    sub = table[table.subj == person].copy()
    sub["score"] = [model.score(r) for _, r in sub.iterrows()]
    all_scores[person] = sub
fig, axes = plt.subplots(4, 2, figsize=(15, 11), sharex=True, sharey=True)
for ax, (person, sub) in zip(axes.flat, all_scores.items()):
    name = sub.rec.iloc[0]; press = data[name][2]
    ax.fill_between(sub.t_end / 60, 0, 1, where=(sub.label == 1), color="orange", alpha=0.3, step="mid")
    ax.plot(sub.t_end / 60, sub.score, "k", lw=1.2); ax.plot(sub.t_end / 60, sub.confidence * 0.1, color="tab:green", lw=1)
    ax.vlines(press / 60, 0.93, 1.0, color="r", lw=0.8); ax.set_title(f"person {person}   AUC {per_person[person]['AUC']:.2f}", fontsize=9)
    ax.set_ylim(0, 1)
axes.flat[-1].axis("off")
for ax in axes[-1]:
    ax.set_xlabel("minutes")
fig.supylabel("drowsiness score (black); signal confidence (green, scaled to 0-0.1)")
plt.tight_layout(); plt.show()

In [13]:
final_model = fit(labelled)
final_model.info = {"auc_unseen_people_mean": float(aucs.mean()), "auc_unseen_people_std": float(aucs.std()), "trained_on": "DD-Database trial 1, subjects 1-7, EOG (LOC-ROC)",
                    "window_s": WINDOW_S, "baseline_s": BASELINE_S, "features": FEATURES}
final_model.save(MODEL_PATH)
print("saved", MODEL_PATH)
display(pd.DataFrame({"weight (per standard deviation of the feature)": final_model.coef}, index=FEATURES).round(2))

saved models/eog_drowsiness_model.json


,weight (per standard deviation of the feature)
rate,-0.29
width,0.76
share,-0.57
ivcv,-0.06


**What the results say (honestly)**
* On people the model has never seen, the score separates "drowsy" from "alert" **better than chance but not strongly**: AUC about 0.65 on average, between 0.57 and 0.74 depending on the person. At the cut-off 0.5 it catches about 60% of the drowsy windows and clears about 67% of the alert ones. It is a **trend indicator** (the black lines rise around the button presses for most people), not a reliable alarm on its own.
* **Reading the weights.** `share` = `rate` x `width`, so the weights cannot be read one at a time. Together they say: relative to the person's own start, **fewer eye events and less time spent inside them mean more drowsy**, with longer individual events partly compensating.
* Why it is not better: only 7 people; the labels are self-reported button presses; one sensor type; and only blink-style features. The EEG (separate notebook, later) is likely to add information.
* Because of this the branch should get a **low trust weight** in the fusion until it has been tested on the college sensor; the mean AUC is saved with the model (`info`) so the fusion can read it.
* **Not claimed:** anything about EMG accuracy. That needs real EMG data (Part B).

# Part B. Live data (the EMG sensor)

Everything below uses the same detector, features, model and output as above. Only three things are sensor-specific, and they are all in the one `EMG_SENSOR` settings block: how the numbers arrive (serial port, Lab Streaming Layer, or a simulation), how to turn them into microvolts, and the mains frequency.

**What EMG is and what to expect.** The sensor picks up the electrical activity of the eyelid muscle (orbicularis oculi): a blink is a short **burst** of 20-250 Hz noise. The front end therefore band-passes, removes the 50 Hz hum, rectifies and smooths, and the same detector finds the bursts. EMG shows when the **muscle works**, not where the eyelid is: a relaxed, drowsy closure gives little signal, a hard blink gives a lot. So blink-rate-style features work, but **closure duration is not reliable** (the output marks `long_closure_reliable: false`, so fusion will not use it for the microsleep rule).

**Not validated on real EMG yet.** The model was trained on EOG events; EMG events are a different kind of event, so the features are taken **relative to the person's own start** and the output says `model_validated_for_this_sensor: false`. When the sensor arrives, do the 3 checks in the last cell.

### Settings for the sensor

In [14]:
EMG_SENSOR = {
    "source": "simulated",         # "simulated" (no sensor needed), "serial" (a board printing numbers) or "lsl" (Lab Streaming Layer)
    "sample_rate_hz": 500,         # at least 200 (EMG is fast)
    "to_microvolts": 1.0,          # raw number x this = microvolts. ADC counts: volts_per_count x 1e6 (check the sensor's gain too)
    "offset": 0.0,                 # raw value at rest, e.g. 512 for a 10-bit ADC with a mid-rail offset
    "polarity": 1,                 # -1 if wired the other way round
    "mains_hz": 50.0,              # India / Europe 50, USA 60
    "port": "/dev/cu.usbmodem1101",    # serial: the board prints one number per line, e.g. "512"
    "baudrate": 115200,
    "lsl_stream": "EMG",           # lsl: stream name (or type)
}

In [15]:
class SimulatedSource:
    """Plays a signal against the clock like a connected sensor (speed > 1 only for quick demos)."""

    def __init__(self, times, values, speed=1.0):
        self.times, self.values, self.speed, self.pos = np.asarray(times), np.asarray(values), speed, 0

    def start(self):
        self.t0 = time.time()

    def read(self, now):
        end = int(np.searchsorted(self.times, now, side="right"))
        out = self.times[self.pos:end], self.values[self.pos:end]
        self.pos = max(self.pos, end)
        return out

    def stop(self):
        pass


class SerialSource:
    """A board on a serial port printing one number (or several, comma separated) per line. Uses the sample count and the
    nominal rate for the timestamps, re-anchored to the clock when they drift, so USB batching does not jitter them."""

    def __init__(self, port, baudrate, fs):
        self.port, self.baudrate, self.fs = port, baudrate, fs
        self._rows, self._stop, self._count = [], False, 0

    def start(self):
        import threading
        import serial                                                   # pyserial; the port "loop://" is a test loopback
        self._serial = serial.serial_for_url(self.port, baudrate=self.baudrate, timeout=0.2)
        self._t0 = time.time()
        self._thread = threading.Thread(target=self._reader, daemon=True)
        self._thread.start()

    def _reader(self):
        while not self._stop:
            line = self._serial.readline().decode(errors="ignore").strip()
            try:
                value = float(line.split(",")[0])
            except ValueError:
                continue                                                # half line at start-up or garbage
            now, nominal = time.time() - self._t0, self._count / self.fs
            if abs(now - nominal) > 1.0:
                self._count = int(now * self.fs); nominal = self._count / self.fs
            self._rows.append((nominal, value)); self._count += 1

    def read(self, now=None):
        rows, self._rows = self._rows, []
        return (np.array([r[0] for r in rows]), np.array([r[1] for r in rows])) if rows else (np.empty(0), np.empty(0))

    def stop(self):
        self._stop = True
        self._thread.join(timeout=1.0); self._serial.close()


class LslSource:
    """A Lab Streaming Layer stream (many biosignal devices and OpenBCI can publish one). First sample = time 0."""

    def __init__(self, name):
        self.name, self._first = name, None

    def start(self):
        import pylsl
        found = pylsl.resolve_byprop("name", self.name, 1, 10.0) or pylsl.resolve_byprop("type", self.name, 1, 10.0)
        if not found:
            raise RuntimeError(f"No LSL stream called {self.name!r} found. Is the sensor software streaming?")
        self._inlet = pylsl.StreamInlet(found[0])
        self._inlet.open_stream(10.0)

    def read(self, now=None):
        samples, stamps = self._inlet.pull_chunk(timeout=0.0)
        if not stamps:
            return np.empty(0), np.empty(0)
        self._first = stamps[0] if self._first is None else self._first
        return np.array(stamps) - self._first, np.array(samples, float)[:, 0]

    def stop(self):
        self._inlet.close_stream()


def make_source(sensor):
    if sensor["source"] == "serial":
        return SerialSource(sensor["port"], sensor["baudrate"], sensor["sample_rate_hz"])
    if sensor["source"] == "lsl":
        return LslSource(sensor["lsl_stream"])
    raise ValueError("source must be 'serial' or 'lsl' here (the simulation is built below)")

In [16]:
def run_live(source, monitor, seconds, speed=1.0, plot_every=2.0):
    """Run the monitor against the clock for `seconds` of sensor time; draws the last 30 s and the status. Returns a log (one row per plot)."""
    source.start()
    t0, handle, log, next_plot = time.time(), None, [], 0.0
    try:
        while True:
            now = (time.time() - t0) * speed
            if now > seconds:
                break
            times, values = source.read(now)
            out = monitor.update(times, values) if len(times) else monitor.latest(now)
            if now >= next_plot:
                next_plot += plot_every * speed
                log.append({"t": now, "score": out["score"], "confidence": out["confidence"], "events_per_min": out["raw"].get("events_per_min", np.nan),
                            "status": out["raw"]["status"]})
                fig, axes = plt.subplots(2, 1, figsize=(10, 4.2))
                tr = np.array(monitor.stream.trace)
                if len(tr):
                    keep = tr[:, 0] > now - 20
                    axes[0].plot(tr[keep, 0], tr[keep, 1], lw=0.7)
                    for e in monitor.stream.detector.events:
                        if e["end"] > now - 20: axes[0].axvspan(e["start"], e["end"], color="r", alpha=0.35)
                    axes[0].set_xlim(now - 20, now + 0.5)
                axes[0].set_title(out["raw"]["status"] if out["raw"]["status"] != "ok" else f"t {now:.0f} s   score {out['score']:.2f}   confidence {out['confidence']:.2f}   events/min {out['raw']['events_per_min']:.0f}", fontsize=9)
                axes[0].set_ylabel("activity")
                lg = pd.DataFrame(log)
                axes[1].plot(lg.t, lg.score, "k", label="score"); axes[1].plot(lg.t, lg.confidence, color="tab:green", label="confidence"); axes[1].set_ylim(-0.05, 1.05); axes[1].legend(loc="upper left", fontsize=8)
                buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=65, bbox_inches="tight"); plt.close(fig)
                picture = Image(data=buf.getvalue())
                handle = display(picture, display_id=True) if handle is None else (handle.update(picture) or handle)
            time.sleep(0.03)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        source.stop()
    return pd.DataFrame(log)

**Run it.** With `"source": "simulated"` this plays a **synthetic** EMG recording (known blinks) at 6x speed so the whole live path can be seen without a sensor: the person blinks at a normal rate, then from second 100 at about a fifth of that rate (the drowsy pattern). The demo uses short settings (learning 10 s + 40 s baseline + 30 s window) so it fits in a minute; real use keeps the defaults (30 s + 5 min + 2 min).
Set `EMG_SENSOR["source"]` to `"serial"` or `"lsl"` for the real sensor: the same cell then runs on it, at real speed, with the real defaults.

In [17]:
def synthetic_emg(duration_s, fs, rates, mains_hz=50.0, seed=0):
    """Eyelid-EMG-like signal in microvolts with KNOWN blinks: muscle noise, a burst at every blink, mains hum.
    rates: [(from_second, blinks_per_minute), ...]. Returns times, signal, blink times."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(duration_s * fs)) / fs
    sos = signal.butter(4, [20, min(200, 0.45 * fs)], btype="band", fs=fs, output="sos")
    muscle = lambda: (lambda y: y / y.std())(signal.sosfilt(sos, rng.normal(size=len(t))))
    x = 8 * muscle() + 15 * np.sin(2 * np.pi * mains_hz * t) + 20 * np.sin(2 * np.pi * 0.2 * t)
    carrier, blinks, now = muscle(), [], 12.0
    while now < duration_s - 2:
        rate = [r for s, r in rates if s <= now][-1]
        now += max(1.2, rng.exponential(60 / rate)); blinks.append(now)
    for b in blinks:
        d = rng.uniform(0.08, 0.22)
        shape = np.sin(np.pi / 2 * np.minimum(np.clip((t - (b - d)) / d, 0, 1), np.clip((b + d - t) / d, 0, 1))) ** 2   # smooth bump, half width = d
        x += 150 * rng.uniform(0.8, 1.2) * carrier * shape
    return t, x, np.array(blinks)


fs = EMG_SENSOR["sample_rate_hz"]
if EMG_SENSOR["source"] == "simulated":
    SPEED = 6.0
    sim_t, sim_x, sim_blinks = synthetic_emg(160, fs, [(0, 16), (100, 3)], EMG_SENSOR["mains_hz"], seed=3)
    source = SimulatedSource(sim_t, sim_x, SPEED)
    monitor = DrowsinessMonitor(LinearModel.load(MODEL_PATH), fs, "emg", mains_hz=EMG_SENSOR["mains_hz"], calibrate_s=10, baseline_s=40, window_s=30)
    live_log = run_live(source, monitor, 160, speed=SPEED, plot_every=1.0)
else:
    source = make_source(EMG_SENSOR)
    monitor = DrowsinessMonitor(LinearModel.load(MODEL_PATH), fs, "emg", EMG_SENSOR["to_microvolts"], EMG_SENSOR["offset"], EMG_SENSOR["polarity"], EMG_SENSOR["mains_hz"])
    live_log = run_live(source, monitor, seconds=3600)              # interrupt the kernel to stop
live_log.round(2).tail(8)

,t,score,confidence,events_per_min,status
19,114.04,0.13,1.0,8.0,ok
20,120.21,0.42,1.0,6.0,ok
21,126.16,0.72,1.0,4.0,ok
22,132.14,0.42,1.0,6.0,ok
23,138.13,0.42,1.0,6.0,ok
24,144.10,0.42,1.0,6.0,ok
25,150.06,0.43,1.0,6.0,ok
26,156.17,0.43,1.0,6.0,ok


**Reading the simulated run.** The score stays low while blinks come at the normal rate and rises after second 100 when they become rare; the confidence is 0 while the monitor learns the person's normal activity, then close to 1. This proves the **live code path**; it says nothing about EMG accuracy because the signal is simulated.

### When the sensor arrives: 3 checks (about 10 minutes)
1. **Does it see your blinks?** Run with `source: serial` (or `lsl`) and blink on command 10 times: `events_per_min` and the red marks in the plot should match. If not, check the electrode placement (over the orbicularis oculi, at the outer corner of the eye, reference on the cheekbone), `to_microvolts`, `offset` and `mains_hz`.
2. **Does bad contact lower the confidence?** Pull an electrode off: confidence should drop to 0 within about 2 seconds.
3. **Does the score move the right way?** Hard to test on purpose; record a few real sessions with button presses (like the dataset) and compare, as in Part A. Until then treat the EMG score as unvalidated.

## Self-checks (synthetic signals with known answers)
Quick tests that the code does what it claims, so a wrong result cannot hide behind the real data.

In [18]:
def run_chunks(monitor, times, raw, chunk=0.04):
    n = int(chunk * FS_TEST)
    for i in range(0, len(times), n):
        monitor.update(times[i:i + n], raw[i:i + n])
    return monitor


def found(monitor, truth, tol=0.35):
    mid = np.array([e["start"] + e["duration"] / 2 for e in monitor.stream.detector.events])
    return sum(bool((np.abs(mid - b) < tol).any()) for b in truth), len(mid)


FS_TEST = 500
t_s, x_s, blinks = synthetic_emg(240, FS_TEST, [(0, 15)], 50.0, seed=1)
truth = blinks[blinks > 35]
reference = found(run_chunks(DrowsinessMonitor(None, FS_TEST, "emg"), t_s, x_s), truth)
print(f"EMG: {reference[0]} of {len(truth)} true blinks found, {reference[1]} detections")
assert reference[0] >= 0.95 * len(truth) and reference[1] <= 1.1 * len(truth)

# 1. the result must not depend on the device: another gain, offset, wiring (polarity) give the same events
for gain, offset, polarity in [(0.1, 0, 1), (10.0, 0, 1), (4.88, 512, 1), (2.0, 100, -1)]:
    raw = polarity * x_s / gain + offset                                    # what that device would deliver (counts)
    m = run_chunks(DrowsinessMonitor(None, FS_TEST, "emg", to_microvolts=gain, offset=offset, polarity=polarity), t_s, raw)
    assert found(m, truth) == reference, (gain, offset, polarity)
print("same events for other gain, offset and polarity: OK")

# 2. a flat line, clipping and a lost data stream must drop the confidence; it must recover afterwards
long_t, long_x, _ = synthetic_emg(150, FS_TEST, [(0, 15)], 50.0, seed=2)
for name, damage in {"electrode off (flat line)": lambda v: np.full_like(v, v[0]), "amplifier clipping": lambda v: np.clip(v, -3, 3) * 50,
                     "motion noise": lambda v: v + np.random.default_rng(0).normal(0, 400, len(v))}.items():
    x_bad = long_x.copy(); w = (long_t >= 100) & (long_t < 110); x_bad[w] = damage(x_bad[w])
    conf = {}
    m = DrowsinessMonitor(None, FS_TEST, "emg", calibrate_s=10, baseline_s=20, window_s=30)
    for i in range(0, len(long_t), 20):
        out = m.update(long_t[i:i + 20], x_bad[i:i + 20]); conf[round(long_t[i], 1)] = out["confidence"]
    during, after = conf[109.0], conf[140.0]
    print(f"{name:26s} confidence at 109 s: {during:.2f}   at 140 s: {after:.2f}")
    assert during < 0.3 and after > 0.6, name

m = DrowsinessMonitor(None, FS_TEST, "emg", calibrate_s=10, baseline_s=20, window_s=30)
for i in range(0, 100 * FS_TEST, 20):
    out = m.update(long_t[i:i + 20], long_x[i:i + 20])
assert m.latest(100.0)["confidence"] > 0.6 and m.latest(104.0)["confidence"] == 0.0
print("no data from the sensor for 4 s -> confidence 0: OK")

# 3. the same two checks on REAL EOG: gain, offset and wiring must not change the events found in 10 minutes of subject 1
left, right, _ = data["01M_1"]; seg = slice(0, 600 * FS); t_r = np.arange(600 * FS) / FS
counts = []
for gain, offset, polarity in [(1.0, 0, 1), (0.5, 512, 1), (4.0, 0, -1)]:
    m = DrowsinessMonitor(None, FS, "eog", to_microvolts=gain, offset=offset, polarity=polarity)
    raw = polarity * (left - right)[seg] / gain + offset
    for i in range(0, len(t_r), FS):
        m.update(t_r[i:i + FS], raw[i:i + FS])
    counts.append(len(m.stream.detector.events))
print("events found in 10 min of real EOG with different gain / offset / wiring:", counts)
assert len(set(counts)) == 1
print("ALL CHECKS PASSED")

EMG: 54 of 55 true blinks found, 56 detections


same events for other gain, offset and polarity: OK


electrode off (flat line)  confidence at 109 s: 0.00   at 140 s: 0.94


amplifier clipping         confidence at 109 s: 0.00   at 140 s: 0.94


motion noise               confidence at 109 s: 0.00   at 140 s: 0.94


no data from the sensor for 4 s -> confidence 0: OK


events found in 10 min of real EOG with different gain / offset / wiring: [301, 301, 301]
ALL CHECKS PASSED
